# LLM Evaluations — Spreadsheets

This notebook consolidates every LLM evaluation under `research-files/<diagram>/evaluations/<model>_<format>/` into two long-format CSVs:

1. **Ordinal** (`evaluations_ordinal.csv`) — raw integer scores 1–4 plus justification.
2. **Binarized** (`evaluations_binary.csv`) — same `1`/`3` polarization used by the human ground truth.

### Trial deduplication

Each criterion was evaluated **twice** per (diagram, model, format). We keep only the **latest trial** (highest timestamp in the filename).

### Binarization scheme

LLM scores are integers, so there are no boundary (2.5) cases:

- `score ≤ 2` → **0** (negative)
- `score ≥ 3` → **1** (positive)

### Diagram ID mapping

To match the IDs used in the human ground truth (`gt_scores.csv`), the folder `geoserver` is renamed to `geoserver-cloud` in the output.

## 1. Setup

In [1]:
import json
import re
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
CSV_DIR = NOTEBOOK_DIR.parent
RESEARCH_DIR = CSV_DIR.parent

OUT_ORDINAL = NOTEBOOK_DIR / "evaluations_ordinal.csv"
OUT_BINARY = NOTEBOOK_DIR / "evaluations_binary.csv"
OUT_TRIALS_LOG = NOTEBOOK_DIR / "trials_selected.csv"

DIAGRAM_FOLDERS = [
    "appwrite",
    "geoserver",
    "magda-io",
    "research-modifiability-pattern-experiment",
    "sentry",
    "spryker",
]

DIAGRAM_ID_OVERRIDES = {
    "geoserver": "geoserver-cloud",
}

EXPECTED_CRITERIA = ["accuracy", "clarity", "completeness", "consistency", "level_of_detail"]

print(f"Research dir: {RESEARCH_DIR}")
for d in DIAGRAM_FOLDERS:
    eval_dir = RESEARCH_DIR / d / "evaluations"
    if eval_dir.exists():
        subdirs = sorted(p.name for p in eval_dir.iterdir() if p.is_dir())
        print(f"  {d}: {subdirs}")
    else:
        print(f"  {d}: MISSING evaluations/")

Research dir: C:\Users\João Luiz\Desktop\Trabalhos facul\POC\llm-architecure-assesment\research-files
  appwrite: ['chatgpt_png', 'chatgpt_svg', 'claude_png', 'claude_svg']
  geoserver: ['chatgpt_png', 'chatgpt_svg', 'claude_png', 'claude_svg']
  magda-io: ['chatgpt_png', 'chatgpt_svg', 'claude_png', 'claude_svg']
  research-modifiability-pattern-experiment: ['chatgpt_png', 'chatgpt_xml', 'claude_png', 'claude_xml']
  sentry: ['chatgpt_png', 'chatgpt_xml', 'claude_png', 'claude_xml']
  spryker: ['chatgpt_png', 'chatgpt_xml', 'claude_png', 'claude_xml']


## 2. File discovery and latest-trial selection

For each `<diagram>/evaluations/<model>_<format>/<criterion>_<timestamp>.json`, keep only the file with the highest timestamp per `(diagram, model, format, criterion)`.

In [2]:
FILENAME_RE = re.compile(r"^(?P<criterion>[a-z_]+)_(?P<timestamp>\d{8}_\d{6})\.json$")
SUBDIR_RE = re.compile(r"^(?P<model>[a-z]+)_(?P<format>[a-z]+)$")


def discover_latest_trials() -> pd.DataFrame:
    rows = []
    for folder in DIAGRAM_FOLDERS:
        diagram_id = DIAGRAM_ID_OVERRIDES.get(folder, folder)
        eval_dir = RESEARCH_DIR / folder / "evaluations"
        if not eval_dir.exists():
            continue
        for sub in sorted(eval_dir.iterdir()):
            if not sub.is_dir():
                continue
            sub_match = SUBDIR_RE.match(sub.name)
            if not sub_match:
                print(f"  skipping unknown subdir: {sub.relative_to(RESEARCH_DIR)}")
                continue
            model = sub_match.group("model")
            fmt = sub_match.group("format")
            buckets: dict[str, tuple[str, Path]] = {}
            for f in sub.iterdir():
                if f.suffix != ".json":
                    continue
                m = FILENAME_RE.match(f.name)
                if not m:
                    continue
                crit = m.group("criterion")
                ts = m.group("timestamp")
                prev = buckets.get(crit)
                if prev is None or ts > prev[0]:
                    buckets[crit] = (ts, f)
            for criterion, (ts, path) in buckets.items():
                rows.append(
                    {
                        "Diagram_ID": diagram_id,
                        "Diagram_Folder": folder,
                        "Model": model,
                        "Format": fmt,
                        "Criterion": criterion,
                        "Timestamp": ts,
                        "File": str(path.relative_to(RESEARCH_DIR)),
                    }
                )
    return pd.DataFrame(rows)


trials = discover_latest_trials()
print(f"Latest trials selected: {len(trials)}")
print(f"Distinct (diagram, model, format) combos: "
      f"{trials.groupby(['Diagram_ID', 'Model', 'Format']).ngroups}")
trials.head()

Latest trials selected: 120
Distinct (diagram, model, format) combos: 24


,Diagram_ID,Diagram_Folder,Model,Format,Criterion,Timestamp,File
0,appwrite,appwrite,chatgpt,png,accuracy,20260523_165733,appwrite\evaluations\chatgpt_png\accuracy_2026...
1,appwrite,appwrite,chatgpt,png,clarity,20260523_165757,appwrite\evaluations\chatgpt_png\clarity_20260...
2,appwrite,appwrite,chatgpt,png,completeness,20260523_165818,appwrite\evaluations\chatgpt_png\completeness_...
3,appwrite,appwrite,chatgpt,png,consistency,20260523_165839,appwrite\evaluations\chatgpt_png\consistency_2...
4,appwrite,appwrite,chatgpt,png,level_of_detail,20260523_165901,appwrite\evaluations\chatgpt_png\level_of_deta...


In [3]:
criteria_per_combo = (
    trials.groupby(["Diagram_ID", "Model", "Format"])["Criterion"]
    .apply(lambda s: sorted(s.unique()))
    .reset_index(name="criteria")
)
criteria_per_combo["n_criteria"] = criteria_per_combo["criteria"].str.len()

missing = []
for row in criteria_per_combo.itertuples():
    missing_for_combo = [c for c in EXPECTED_CRITERIA if c not in row.criteria]
    if missing_for_combo:
        missing.append(
            {
                "Diagram_ID": row.Diagram_ID,
                "Model": row.Model,
                "Format": row.Format,
                "missing_criteria": missing_for_combo,
            }
        )

if missing:
    print("WARNING - combos with missing criteria:")
    display(pd.DataFrame(missing))
else:
    print(f"OK - every (diagram, model, format) has all {len(EXPECTED_CRITERIA)} criteria")
criteria_per_combo

OK - every (diagram, model, format) has all 5 criteria


,Diagram_ID,Model,Format,criteria,n_criteria
0,appwrite,chatgpt,png,"[accuracy, clarity, completeness, consistency,...",5
1,appwrite,chatgpt,svg,"[accuracy, clarity, completeness, consistency,...",5
2,appwrite,claude,png,"[accuracy, clarity, completeness, consistency,...",5
3,appwrite,claude,svg,"[accuracy, clarity, completeness, consistency,...",5
4,geoserver-cloud,chatgpt,png,"[accuracy, clarity, completeness, consistency,...",5
5,geoserver-cloud,chatgpt,svg,"[accuracy, clarity, completeness, consistency,...",5
6,geoserver-cloud,claude,png,"[accuracy, clarity, completeness, consistency,...",5
7,geoserver-cloud,claude,svg,"[accuracy, clarity, completeness, consistency,...",5
8,magda-io,chatgpt,png,"[accuracy, clarity, completeness, consistency,...",5
9,magda-io,chatgpt,svg,"[accuracy, clarity, completeness, consistency,...",5


## 3. Parse JSON files

Reads each selected JSON, extracts the score and justification for every guideline. The `score` field looks like `"3 (Good)"` — we parse out the leading integer.

In [4]:
SCORE_RE = re.compile(r"^\s*(\d+)")


def parse_score(raw: str) -> int:
    m = SCORE_RE.match(raw)
    if not m:
        raise ValueError(f"Could not parse score from {raw!r}")
    value = int(m.group(1))
    if value not in (1, 2, 3, 4):
        raise ValueError(f"Score out of expected range 1-4: {raw!r}")
    return value


def parse_trial(file_path: Path) -> list[dict]:
    payload = json.loads(file_path.read_text(encoding="utf-8"))
    parsed = payload.get("parsed_response")
    if not parsed:
        raise ValueError(f"Missing parsed_response in {file_path}")
    rows = []
    criterion = parsed.get("criterion")
    for g in parsed.get("guidelines", []):
        rows.append(
            {
                "criterion": criterion,
                "Guideline": g["id"],
                "score": parse_score(g["score"]),
                "score_raw": g["score"],
                "justification": g.get("justification", ""),
            }
        )
    return rows


records = []
for trial in trials.itertuples():
    abs_path = RESEARCH_DIR / trial.File
    try:
        for parsed in parse_trial(abs_path):
            if parsed["criterion"] != trial.Criterion:
                print(
                    f"WARN: criterion mismatch in {trial.File}: "
                    f"filename says {trial.Criterion!r}, JSON says {parsed['criterion']!r}"
                )
            records.append(
                {
                    "Diagram_ID": trial.Diagram_ID,
                    "Model": trial.Model,
                    "Format": trial.Format,
                    "Criterion": trial.Criterion,
                    "Guideline": parsed["Guideline"],
                    "score": parsed["score"],
                    "score_raw": parsed["score_raw"],
                    "justification": parsed["justification"],
                    "Trial_Timestamp": trial.Timestamp,
                }
            )
    except Exception as exc:
        print(f"ERROR parsing {trial.File}: {exc}")

evaluations_long = pd.DataFrame(records)
print(f"Parsed rows: {len(evaluations_long)}")
print(f"Unique (diagram, model, format, criterion, guideline) keys: "
      f"{evaluations_long.groupby(['Diagram_ID','Model','Format','Criterion','Guideline']).ngroups}")
evaluations_long.head()

Parsed rows: 360
Unique (diagram, model, format, criterion, guideline) keys: 360


,Diagram_ID,Model,Format,Criterion,Guideline,score,score_raw,justification,Trial_Timestamp
0,appwrite,chatgpt,png,accuracy,component_matching,4,4 (Excellent),Diagram shows a monolithic API surface (REST/G...,20260523_165733
1,appwrite,chatgpt,png,accuracy,interaction_matching,3,3 (Good),Primary flows match: clients → load balancer (...,20260523_165733
2,appwrite,chatgpt,png,clarity,element_identification,3,3 (Good),"Major components are named (clients, Loadbalan...",20260523_165757
3,appwrite,chatgpt,png,clarity,reading_flow,3,3 (Good),There is a clear top-to-bottom path from clien...,20260523_165757
4,appwrite,chatgpt,png,clarity,symbol_label_readability,3,3 (Good),Elements are visually distinct and mostly read...,20260523_165757


In [5]:
dupes = (
    evaluations_long.groupby(["Diagram_ID", "Model", "Format", "Criterion", "Guideline"])
    .size()
    .reset_index(name="n")
)
bad = dupes[dupes["n"] != 1]
print(f"Duplicate keys (should be 0): {len(bad)}")
if len(bad):
    display(bad)

guidelines_per_combo = (
    evaluations_long.groupby(["Diagram_ID", "Model", "Format"])["Guideline"]
    .nunique()
    .reset_index(name="n_guidelines")
)
display(guidelines_per_combo)

Duplicate keys (should be 0): 0


,Diagram_ID,Model,Format,n_guidelines
0,appwrite,chatgpt,png,15
1,appwrite,chatgpt,svg,15
2,appwrite,claude,png,15
3,appwrite,claude,svg,15
4,geoserver-cloud,chatgpt,png,15
5,geoserver-cloud,chatgpt,svg,15
6,geoserver-cloud,claude,png,15
7,geoserver-cloud,claude,svg,15
8,magda-io,chatgpt,png,15
9,magda-io,chatgpt,svg,15


## 4. Ordinal table

In [6]:
evaluations_ordinal = (
    evaluations_long[
        [
            "Diagram_ID",
            "Model",
            "Format",
            "Criterion",
            "Guideline",
            "score",
            "justification",
            "Trial_Timestamp",
        ]
    ]
    .sort_values(["Diagram_ID", "Model", "Format", "Criterion", "Guideline"])
    .reset_index(drop=True)
)
evaluations_ordinal

,Diagram_ID,Model,Format,Criterion,Guideline,score,justification,Trial_Timestamp
0,appwrite,chatgpt,png,accuracy,component_matching,4,Diagram shows a monolithic API surface (REST/G...,20260523_165733
1,appwrite,chatgpt,png,accuracy,interaction_matching,3,Primary flows match: clients → load balancer (...,20260523_165733
2,appwrite,chatgpt,png,clarity,element_identification,3,"Major components are named (clients, Loadbalan...",20260523_165757
3,appwrite,chatgpt,png,clarity,reading_flow,3,There is a clear top-to-bottom path from clien...,20260523_165757
4,appwrite,chatgpt,png,clarity,specific_symbology,2,Only a few technology-specific cues appear (Do...,20260523_165757
...,...,...,...,...,...,...,...,...
355,spryker,claude,xml,consistency,notation_uniformity,3,The diagram consistently uses rounded rectangl...,20260522_180559
356,spryker,claude,xml,consistency,technology_symbols,2,Different technologies are distinguished by fi...,20260522_180559
357,spryker,claude,xml,level_of_detail,element_detail,3,"Components are named (DE/AT/US nginx, PHP FPM,...",20260522_180617
358,spryker,claude,xml,level_of_detail,interaction_detail,2,"Edges show connectivity between components, in...",20260522_180617


## 5. Binarized table

Same `0`/`1` scheme used for the human ground truth, but since LLM scores are integers there are no boundary (2.5) cases.

- `score ≤ 2` → `0` (negative)
- `score ≥ 3` → `1` (positive)

In [7]:
LOW_LABEL = 0
HIGH_LABEL = 1

evaluations_binary = evaluations_ordinal[
    ["Diagram_ID", "Model", "Format", "Criterion", "Guideline"]
].copy()
evaluations_binary["llm_label"] = evaluations_ordinal["score"].apply(
    lambda s: LOW_LABEL if s <= 2 else HIGH_LABEL
)

print("Binary label distribution:")
print(evaluations_binary["llm_label"].value_counts().sort_index().to_string())
evaluations_binary

Binary label distribution:
llm_label
0    128
1    232


,Diagram_ID,Model,Format,Criterion,Guideline,llm_label
0,appwrite,chatgpt,png,accuracy,component_matching,1
1,appwrite,chatgpt,png,accuracy,interaction_matching,1
2,appwrite,chatgpt,png,clarity,element_identification,1
3,appwrite,chatgpt,png,clarity,reading_flow,1
4,appwrite,chatgpt,png,clarity,specific_symbology,0
...,...,...,...,...,...,...
355,spryker,claude,xml,consistency,notation_uniformity,1
356,spryker,claude,xml,consistency,technology_symbols,0
357,spryker,claude,xml,level_of_detail,element_detail,1
358,spryker,claude,xml,level_of_detail,interaction_detail,0


## 6. Export results

In [8]:
evaluations_ordinal.to_csv(OUT_ORDINAL, index=False)
evaluations_binary.to_csv(OUT_BINARY, index=False)
trials.to_csv(OUT_TRIALS_LOG, index=False)

print("Generated files:")
print(f"  {OUT_ORDINAL.name}      ({len(evaluations_ordinal)} rows)")
print(f"  {OUT_BINARY.name}       ({len(evaluations_binary)} rows)")
print(f"  {OUT_TRIALS_LOG.name}      ({len(trials)} rows)")

Generated files:
  evaluations_ordinal.csv      (360 rows)
  evaluations_binary.csv       (360 rows)
  trials_selected.csv      (120 rows)
